# Layered Random Clifford Circuit

Generate a compact random Clifford circuit, then insert a random single-qubit Pauli layer between every pair of its original layers. A fixed seed keeps the example reproducible.

In [1]:
import numpy as np

from sympleq.core.circuits import Gate, GATES

rng = np.random.default_rng(7)
dimensions = [2, 2]
random_clifford = Gate.from_random(2, 2, num_transvections=3, rng=rng)
circuit = random_clifford.to_circuit(dimensions)
original_layer_count = circuit.n_layers()

print(f'Random Clifford: {original_layer_count} layers')
print(circuit.gates_layout(with_qudit_indices=True, wrap=False))

Random Clifford: 8 layers
         ┌─────┐  ┌─────┐  ┌─────┐  ┌─────┐  ┌─────┐  ┌─────┐  ┌─────┐  ┌─────┐     
 0:   ===│  H  │==│  H* │==│  CX │==│  H  │==│  CX │==│  H  │==│  H* │==│Pauli│===  
         └─────┘  └─────┘  └──┬──┘  └─────┘  └──┬──┘  └─────┘  └─────┘  └──┬──┘     
         ┌─────┐  ┌─────┐  ┌──┴──┐           ┌──┴──┐                    ┌──┴──┐     
 1:   ===│  H  │==│  H* │==│  CX │===========│  CX │====================│Pauli│===  
         └─────┘  └─────┘  └─────┘           └─────┘                    └─────┘     


In [2]:
pauli_gates = (GATES.X, GATES.Y, GATES.Z)

# Insert from right to left so later insertions do not change earlier positions.
for position in range(original_layer_count - 1, 0, -1):
    gates = [pauli_gates[rng.integers(len(pauli_gates))] for _ in dimensions]
    qudit_indices = [(qudit,) for qudit in range(len(dimensions))]
    circuit.insert_layer(position, gates, qudit_indices)

print(f'Circuit with Pauli layers: {circuit.n_layers()} layers')
print(circuit.gates_layout(with_qudit_indices=True, wrap=False))

Circuit with Pauli layers: 15 layers
         ┌─────┐  ┌─────┐  ┌─────┐  ┌─────┐  ┌─────┐  ┌─────┐  ┌─────┐  ┌─────┐  ┌─────┐  ┌─────┐  ┌─────┐  ┌─────┐  ┌─────┐  ┌─────┐  ┌─────┐     
 0:   ===│  H  │==│  Z  │==│  H* │==│  Y  │==│  CX │==│  Z  │==│  H  │==│  X  │==│  CX │==│  X  │==│  H  │==│  Y  │==│  H* │==│  Z  │==│Pauli│===  
         └─────┘  └─────┘  └─────┘  └─────┘  └──┬──┘  └─────┘  └─────┘  └─────┘  └──┬──┘  └─────┘  └─────┘  └─────┘  └─────┘  └─────┘  └──┬──┘     
         ┌─────┐  ┌─────┐  ┌─────┐  ┌─────┐  ┌──┴──┐  ┌─────┐  ┌─────┐           ┌──┴──┐  ┌─────┐  ┌─────┐  ┌─────┐                    ┌──┴──┐     
 1:   ===│  H  │==│  X  │==│  H* │==│  X  │==│  CX │==│  X  │==│  Y  │===========│  CX │==│  Z  │==│  Z  │==│  X  │====================│Pauli│===  
         └─────┘  └─────┘  └─────┘  └─────┘  └─────┘  └─────┘  └─────┘           └─────┘  └─────┘  └─────┘  └─────┘                    └─────┘     
